In [4]:
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa, librosa.display
import matplotlib.pyplot as plt

def mmss_to_seconds(mmss: str) -> float:
    mm, ss = mmss.strip().split(":")
    return int(mm) * 60 + float(ss)

def load_clip_centered(wav_path: Path, center_sec: float, clip_sec: float = 5.0):
    wav_path = Path(wav_path)

    with sf.SoundFile(str(wav_path)) as f:
        sr = f.samplerate
        total_sec = len(f) / sr

        start_sec = max(0.0, center_sec - clip_sec / 2)
        # clamp so we don’t run past end of file
        start_sec = min(start_sec, max(0.0, total_sec - clip_sec))

        start_frame = int(start_sec * sr)
        n_frames = int(clip_sec * sr)

        f.seek(start_frame)
        x = f.read(n_frames, dtype="float32", always_2d=True)
        x = x.mean(axis=1)  # mono

    return x, sr, start_sec

def save_spectrogram_png(y, sr, out_png: Path, title: str):
    # nicer spectrogram: use mel
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=256, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, hop_length=256, x_axis="time", y_axis="mel")
    plt.colorbar(format="%+2.0f dB")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()

def export_clip_and_spec(wav_path: Path, time_mmss: str, clip_sec: float = 5.0, out_dir: Path = Path("manual_exports")):
    out_dir = Path(out_dir)
    (out_dir / "clips").mkdir(parents=True, exist_ok=True)
    (out_dir / "specs").mkdir(parents=True, exist_ok=True)

    center_sec = mmss_to_seconds(time_mmss)
    y, sr, start_sec = load_clip_centered(wav_path, center_sec, clip_sec=clip_sec)

    stem = Path(wav_path).stem
    safe_mmss = time_mmss.replace(":", "-")
    clip_path = out_dir / "clips" / f"{stem}_center{safe_mmss}_{clip_sec:.0f}s.wav"
    png_path  = out_dir / "specs" / f"{stem}_center{safe_mmss}_{clip_sec:.0f}s.png"

    sf.write(str(clip_path), y, sr)
    save_spectrogram_png(y, sr, png_path, title=f"{Path(wav_path).name} (center {time_mmss}, {clip_sec:.0f}s)")

    return clip_path, png_path, start_sec


In [5]:
wav = Path("/Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53/02-11-25 to 02-25-25/20250211/20250211_190000.WAV")
clip_path, png_path, start_sec = export_clip_and_spec(wav, "37:23", clip_sec=5.0)

print("Saved clip:", clip_path)
print("Saved spectrogram:", png_path)
print("Clip starts at (sec):", start_sec)


Saved clip: manual_exports/clips/20250211_190000_center37-23_5s.wav
Saved spectrogram: manual_exports/specs/20250211_190000_center37-23_5s.png
Clip starts at (sec): 2240.5


In [6]:
wav = Path("/Volumes/aid_elephants_interaction/Audio Data/2025/06_FR57/02-11-25 to 02-25-25/20250211/20250211_190000.WAV")
clip_path, png_path, start_sec = export_clip_and_spec(wav, "37:23", clip_sec=5.0)

print("Saved clip:", clip_path)
print("Saved spectrogram:", png_path)
print("Clip starts at (sec):", start_sec)

Saved clip: manual_exports/clips/20250211_190000_center37-23_5s.wav
Saved spectrogram: manual_exports/specs/20250211_190000_center37-23_5s.png
Clip starts at (sec): 2240.5
